In [ ]:
# 1) GPU check + package installs
import sys
print(sys.version)

!nvidia-smi
!pip -q install pretty_midi mido tqdm

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false
# 2) All imports + config values (copied from src/config.py)
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42
MIDI_MIN_PITCH = 21
MIDI_MAX_PITCH = 108
PITCH_DIM = MIDI_MAX_PITCH - MIDI_MIN_PITCH + 1

STEPS_PER_BAR = 16
BEATS_PER_BAR = 4
STEPS_PER_BEAT = STEPS_PER_BAR // BEATS_PER_BAR
DEFAULT_TEMPO_BPM = 120.0

SEQUENCE_LENGTH = 256
WINDOW_STEP = 128

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
VALIDATION_SPLIT = 0.2
NUM_WORKERS = 0

AE_HIDDEN_SIZE = 256
AE_NUM_LAYERS = 2
AE_LATENT_DIM = 64
AE_DROPOUT = 0.2

DATASET_PATH = Path('/kaggle/input/datasets/jackvial/themaestrodatasetv2')
OUTPUT_ROOT = Path('/kaggle/working/outputs')
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
GENERATED_MIDI_DIR = OUTPUT_ROOT / 'generated_midis'

for directory in [OUTPUT_ROOT, CHECKPOINTS_DIR, PLOTS_DIR, GENERATED_MIDI_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Your attached Kaggle input path can be one of these, depending on how it is mounted.
CODE_INPUT_CANDIDATES = [
    Path('/kaggle/input/music-generation-unsupervised-task1-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task1-src'),
]

code_input_root = None
for candidate in CODE_INPUT_CANDIDATES:
    if candidate.exists():
        code_input_root = candidate
        break

if code_input_root is None:
    raise FileNotFoundError(
        'Code dataset path not found. Run "!ls /kaggle/input" and update CODE_INPUT_CANDIDATES.'
    )

if (code_input_root / 'music-generation-unsupervised').exists():
    code_input_root = code_input_root / 'music-generation-unsupervised'

PROJECT_ROOT = Path('/kaggle/working/music-generation-unsupervised')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(code_input_root, PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Code input used:', code_input_root)
print('Project root exists:', PROJECT_ROOT.exists())

In [ ]:
# pyright: reportMissingImports=false
# 3) Data loading and preprocessing
from torch.utils.data import DataLoader

from src.preprocessing.midi_parser import discover_midi_files
from src.preprocessing.piano_roll import load_windowed_piano_rolls
from src.training.train_ae import PianoRollWindowDataset, split_midi_files

midi_files = discover_midi_files(DATASET_PATH)
print(f'Total MIDI files discovered: {len(midi_files)}')

train_files, validation_files = split_midi_files(
    midi_files,
    validation_split=VALIDATION_SPLIT,
    seed=SEED,
    split_name='task1',
)

train_windows = load_windowed_piano_rolls(train_files, sequence_length=SEQUENCE_LENGTH, step_size=WINDOW_STEP)
validation_windows = load_windowed_piano_rolls(validation_files, sequence_length=SEQUENCE_LENGTH, step_size=WINDOW_STEP)

train_dataset = PianoRollWindowDataset(train_windows)
validation_dataset = PianoRollWindowDataset(validation_windows)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f'Train windows: {len(train_dataset)}')
print(f'Validation windows: {len(validation_dataset)}')

In [ ]:
# 4) Model definition (from src/models/autoencoder.py)
from src.models.autoencoder import LSTMAutoencoder

model = LSTMAutoencoder(
    input_dim=PITCH_DIM,
    hidden_size=AE_HIDDEN_SIZE,
    num_layers=AE_NUM_LAYERS,
    latent_dim=AE_LATENT_DIM,
    dropout=AE_DROPOUT,
).to(DEVICE)

model

In [ ]:
# 5) Training loop (from src/training/train_ae.py)
from src.training.train_ae import train_autoencoder

results = train_autoencoder(
    train_loader=train_loader,
    validation_loader=validation_loader,
    model=model,
)

print('Best checkpoint saved at:', results['checkpoint_path'])
print('Last train loss:', results['train_losses'][-1])
print('Last validation loss:', results['validation_losses'][-1])

In [ ]:
# 6) Loss plot save -> /kaggle/working/outputs/plots/task1_loss_curve.png
loss_plot_path = PLOTS_DIR / 'task1_loss_curve.png'

plt.figure(figsize=(10, 5))
plt.plot(results['train_losses'], label='Train Loss')
plt.plot(results['validation_losses'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Task 1 - LSTM Autoencoder Loss Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(loss_plot_path, dpi=200)
plt.show()

print('Saved:', loss_plot_path)

In [ ]:
# 7) Music generation -> save MIDIs to /kaggle/working/outputs/generated_midis/
from src.generation.generate_music import generate_task1_samples

generated_paths = generate_task1_samples(
    checkpoint_path=results['checkpoint_path'],
    num_samples=5,
    output_dir=GENERATED_MIDI_DIR,
    tempo_bpm=DEFAULT_TEMPO_BPM,
)

for path in generated_paths:
    print(path)

In [ ]:
# 8) Summary cell printing all deliverables
expected_files = [
    PLOTS_DIR / 'task1_loss_curve.png',
] + [GENERATED_MIDI_DIR / f'task1_sample_{index}.mid' for index in range(1, 6)]

print('Task 1 Deliverables Summary')
for file_path in expected_files:
    status = 'OK' if file_path.exists() else 'MISSING'
    print(f'[{status}] {file_path}')